In [3]:
import mne
import numpy as np
import pandas as pd

from ieeg_prep.task_analysis import (
    DEFAULT_EVENT_CODES,
    get_trial_word_boundaries_from_block,
    compute_response_vector,
    permutation_test,
    load_block
)

In [4]:
envelope_fif_path="/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0636/preprocessed/preprocessed_envelope_ieeg.fif"
events_path= "/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0636/preprocessed/preprocessed_events.npy"
blocks_info_path="/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0636/preprocessed/blocks_info.json"
electrodes_csv_path="/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0636/preprocessed/electrodes.csv"

In [5]:
event_codes = DEFAULT_EVENT_CODES
n_words = 12

In [17]:
events = np.load(events_path)
ind_list = np.arange(events.shape[0])
start_ind = ind_list[events[:,2]==1]
end_ind = ind_list[events[:,2]==2]
start_ind[0]

np.int64(0)

In [24]:
for start, end in zip(start_ind, end_ind):
    block_events = events[start:end+1,2] 
    print(np.shape(block_events))
    print(np.sum(block_events == 5))
    print(events[start:end+1] )

(86,)
5
[[ 8627     0     1]
 [ 8755     0     6]
 [12116     0     5]
 [12123     0     8]
 [12140     0     4]
 [12272     0     4]
 [12404     0     4]
 [12536     0     4]
 [12668     0     4]
 [12800     0     4]
 [12932     0     4]
 [13064     0     4]
 [13196     0     4]
 [13328     0     4]
 [13460     0     4]
 [13592     0     4]
 [13724     0     9]
 [13820     0     8]
 [13844     0     5]
 [13850     0     8]
 [13868     0     4]
 [13993     0     7]
 [14000     0     4]
 [14132     0     4]
 [14264     0     4]
 [14396     0     4]
 [14528     0     4]
 [14660     0     4]
 [14792     0     4]
 [14924     0     4]
 [15056     0     4]
 [15188     0     4]
 [15320     0     4]
 [15452     0     9]
 [15548     0     8]
 [15573     0     5]
 [15580     0     8]
 [15596     0     4]
 [15694     0     7]
 [15728     0     4]
 [15860     0     4]
 [15992     0     4]
 [16124     0     4]
 [16256     0     4]
 [16388     0     4]
 [16520     0     4]
 [16652     0     4]
 [167

In [6]:
env_raw = mne.io.read_raw_fif(str(envelope_fif_path), preload=True, verbose=False)
bad_ch_set = set(env_raw.info["bads"])

all_picks = mne.pick_types(env_raw.info, misc=True, exclude=[])
all_ch_names = [env_raw.ch_names[i] for i in all_picks]
seeg_names = [ch.replace("_hg", "") for ch in all_ch_names]
bad_seeg_names = [ch.replace("_hg", "") for ch in all_ch_names if ch in bad_ch_set]

envelope_data = env_raw.get_data(picks=all_picks)  # (n_channels, n_samples)


per_block: dict[str, dict] = {}
respvecs: list[np.ndarray] = []
respvec_ideals: list[np.ndarray] = []

FileNotFoundError: fname does not exist: "/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0636/preprocessed/preprocessed_envelope_ieeg.fif"

In [5]:
langloc_blocks = ["langloc1","langloc2"]

In [6]:
for label in langloc_blocks:
        print(f"Processing block: {label}")
        block = load_block(blocks_info_path, label, events)
        trials, bad_trials = get_trial_word_boundaries_from_block(
            block, event_codes=event_codes, n_words=n_words
        )
        if bad_trials:
            print(f"  Warning: {len(bad_trials)} malformed trial(s) skipped in '{label}'")

Processing block: langloc1
Processing block: langloc2


In [7]:
events

array([[  1030,      0,      1],
       [  1061,      0,      6],
       [  1662,      0,      3],
       ...,
       [323810,      0,      4],
       [324711,      0,      4],
       [325180,      0,      5]], shape=(4693, 3))

In [8]:
sum(events[:,0]>-1)

np.int64(4693)

In [9]:
trials

[{'block_start_event_idx': 1126,
  'trial_number': 0,
  'trial_start_event_idx_local': 2,
  'trial_start_event_idx_global': 1128,
  'word_event_indices_local': [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
  'word_event_indices_global': [1130,
   1131,
   1132,
   1133,
   1134,
   1135,
   1136,
   1137,
   1138,
   1139,
   1140,
   1141],
  'word_samples': array([110046, 110079, 110112, 110145, 110178, 110211, 110244, 110277,
         110310, 110343, 110376, 110409]),
  'word_codes': array([4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]),
  'probe_event_idx_local': 16,
  'probe_event_idx_global': 1142,
  'probe_sample': 110442,
  'word_bounds': [(110046, 110079),
   (110079, 110112),
   (110112, 110145),
   (110145, 110178),
   (110178, 110211),
   (110211, 110244),
   (110244, 110277),
   (110277, 110310),
   (110310, 110343),
   (110343, 110376),
   (110376, 110409),
   (110409, 110442)]},
 {'block_start_event_idx': 1126,
  'trial_number': 1,
  'trial_start_event_idx_local': 19,
  'trial_start

In [10]:
langloc_blocks = []
for idx, block in enumerate(complete_blocks):
    num_trials = np.sum(np.asarray(block["events"]) == 6)
    print(f"Block {idx}: {num_trials} trials")
    num_nw_conds = np.sum(np.asarray(block["events"]) == 4)
    print(f"Block {idx}: {num_nw_conds} non word conditions")
    if num_trials == 48:
        langloc_blocks.append(block)
        

NameError: name 'complete_blocks' is not defined

In [44]:
ied = np.load("/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0004/processed/ied_results.npz", allow_pickle=True)

In [45]:
 coordinates_csv= "/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0004/tasks_raw/sub-EMOP0004_ses-ieeg1_atlas-CerebrAThomasMiddle_electrodes.csv"

In [46]:
ied["channel_spike_rates"]

array([1.88452422e+00, 1.34451435e+00, 6.06133520e-01, 8.70628147e-01,
       3.96741940e-01, 2.97556455e-01, 2.75515236e-01, 2.64494627e-01,
       1.24532887e+00, 5.28989254e-01, 1.33349374e+00, 6.17154130e-01,
       4.40824378e-01, 7.05319005e-01, 4.18783159e-01, 5.95112911e-01,
       3.30618284e-01, 3.08577065e-01, 2.75515236e-01, 3.19597674e-01,
       3.19597674e-01, 1.54288532e-01, 1.32247313e-01, 2.09391580e-01,
       3.08577065e-01, 2.31432799e-01, 4.40824378e-01, 3.19597674e-01,
       8.37566319e-01, 3.74700722e-01, 3.19597674e-01, 2.64494627e-01,
       2.20412189e-01, 1.85146239e+00, 8.48586928e-01, 1.10206095e-01,
       1.43267923e-01, 2.31432799e-01, 1.19022582e+00, 1.96166848e+00,
       6.17154130e-01, 5.73071692e-01, 5.17968645e-01, 2.09391580e-01,
       9.03689976e-01, 7.16339615e-01, 3.41638893e-01, 2.53474018e-01,
       2.42453408e-01, 2.75515236e-01, 3.08577065e-01, 8.37566319e-01,
       1.10206095e-01, 1.87350361e-01, 1.10206095e-01, 6.50215958e-01,
      

In [47]:
ied_ch_names = [str(ch) for ch in ied["ch_names"]]

In [48]:
spike_counts = dict(zip(ied_ch_names, ied["channel_spike_counts"]))
spike_rates = dict(zip(ied_ch_names, ied["channel_spike_rates"]))

In [49]:
spike_counts

{'LAm1': np.int64(171),
 'LAm2': np.int64(122),
 'LAm3': np.int64(55),
 'LAm4': np.int64(79),
 'LAm5': np.int64(36),
 'LAm6': np.int64(27),
 'LAm7': np.int64(25),
 'LAm8': np.int64(24),
 'LAm9': np.int64(113),
 'LAm10': np.int64(48),
 'LAm11': np.int64(121),
 'LAm12': np.int64(56),
 'LAm13': np.int64(40),
 'LAHc1': np.int64(64),
 'LAHc2': np.int64(38),
 'LAHc3': np.int64(54),
 'LAHc4': np.int64(30),
 'LAHc5': np.int64(28),
 'LAHc6': np.int64(25),
 'LAHc7': np.int64(29),
 'LAHc8': np.int64(29),
 'LAHc9': np.int64(14),
 'LAHc10': np.int64(12),
 'LAHc11': np.int64(19),
 'LAHc12': np.int64(28),
 'LAHc13': np.int64(21),
 'LMiHc1': np.int64(40),
 'LMiHc2': np.int64(29),
 'LMiHc3': np.int64(76),
 'LMiHc4': np.int64(34),
 'LMiHc5': np.int64(29),
 'LMiHc6': np.int64(24),
 'LMiHc7': np.int64(20),
 'LMiHc8': np.int64(168),
 'LMiHc9': np.int64(77),
 'LMiHc10': np.int64(10),
 'LMiHc11': np.int64(13),
 'LMiHc12': np.int64(21),
 'LTePo1': np.int64(108),
 'LTePo2': np.int64(178),
 'LTePo3': np.int64(5

In [50]:
import pandas as pd
df = pd.read_csv(coordinates_csv)

In [51]:
import re
def _normalize_label(label: str) -> str:
    """Normalize electrode label to MNE channel name format (e.g. 'LAm-01' -> 'LAm1')."""
    m = re.match(r"([A-Za-z]+)-?0*([0-9]+)$", str(label))
    if m:
        return f"{m.group(1)}{int(m.group(2))}"
    return label

In [22]:
df

,label,atlas_label,GM,WM,CSF,x,y,z,mni_x,mni_y,mni_z
0,LAm-01,Hippocampus,0.992,0.992,0.002,-13.584,-20.118,37.439,-14.200,-9.554,-16.098
1,LAm-02,Amygdala,0.986,0.986,0.006,-16.990,-19.842,36.876,-17.839,-9.098,-16.606
2,LAm-03,Hippocampus,0.953,0.953,0.029,-20.441,-19.661,36.043,-21.572,-8.669,-17.590
3,LAm-04,Inferior Lateral Ventricle,0.975,0.975,0.013,-23.646,-19.522,35.343,-25.147,-8.468,-18.776
4,LAm-05,Inferior Lateral Ventricle,0.743,0.743,0.196,-27.093,-19.374,34.594,-28.865,-8.254,-20.670
...,...,...,...,...,...,...,...,...,...,...,...
303,RAIn-14,Caudal Middle Frontal,0.978,0.978,0.001,33.237,6.901,84.648,35.323,23.555,39.768
304,RAIn-15,Caudal Middle Frontal,0.991,0.991,0.002,33.637,7.502,88.022,36.841,23.857,42.726
305,RAIn-16,Caudal Middle Frontal,0.992,0.992,0.002,33.889,8.305,91.588,37.661,24.691,45.979
306,RAIn-17,Caudal Middle Frontal,0.992,0.992,0.002,34.109,8.994,94.810,37.421,26.009,49.111


In [23]:
df["_normalized"] = df["label"].apply(_normalize_label)

In [27]:
import json

with open("/storage/cedar/cedar_cos/cos-lab-aivanova7/eayesh3/iEEG/EMOP0004/preprocessed/preprocessed_metadata.json",'r') as f:
    metadata = json.load(f)

seeg_set = set(metadata["ch_names_seeg"])

In [28]:
df = df[df["_normalized"].isin(seeg_set)].copy()
df

,label,atlas_label,GM,WM,CSF,x,y,z,mni_x,mni_y,mni_z,_normalized
0,LAm-01,Hippocampus,0.992,0.992,0.002,-13.584,-20.118,37.439,-14.200,-9.554,-16.098,LAm1
1,LAm-02,Amygdala,0.986,0.986,0.006,-16.990,-19.842,36.876,-17.839,-9.098,-16.606,LAm2
2,LAm-03,Hippocampus,0.953,0.953,0.029,-20.441,-19.661,36.043,-21.572,-8.669,-17.590,LAm3
3,LAm-04,Inferior Lateral Ventricle,0.975,0.975,0.013,-23.646,-19.522,35.343,-25.147,-8.468,-18.776,LAm4
4,LAm-05,Inferior Lateral Ventricle,0.743,0.743,0.196,-27.093,-19.374,34.594,-28.865,-8.254,-20.670,LAm5
...,...,...,...,...,...,...,...,...,...,...,...,...
297,RAIn-08,Pars Opercularis,0.991,0.991,0.001,31.792,2.965,63.811,37.274,16.456,13.313,RAIn8
298,RAIn-09,Pars Opercularis,0.985,0.985,0.001,32.051,3.623,67.638,36.342,16.308,18.390,RAIn9
299,RAIn-10,Pars Opercularis,0.986,0.986,0.001,32.305,4.387,71.150,35.028,17.997,22.721,RAIn10
300,RAIn-11,Rostral Middle Frontal,0.992,0.992,0.002,32.476,4.943,74.308,34.498,19.582,27.152,RAIn11


In [29]:
bad_set = set(metadata["bad_channels"])

In [30]:
df["is_bad"] = df["_normalized"].apply(lambda ch: int(ch in bad_set))

In [33]:
df[210:230]

,label,atlas_label,GM,WM,CSF,x,y,z,mni_x,mni_y,mni_z,_normalized,is_bad
252,RANT-14,Superior Temporal,0.990,0.990,0.004,46.882,-17.208,45.733,53.114,-6.295,-1.322,RANT14,0
253,RANT-15,Superior Temporal,0.907,0.907,0.001,50.406,-17.637,44.458,57.367,-6.669,-2.752,RANT15,0
254,RANT-16,Superior Temporal,0.958,0.958,0.025,53.766,-17.687,43.414,62.217,-7.537,-4.536,RANT16,0
257,RPul-01,Pul,0.985,0.985,0.007,6.062,-42.450,54.509,8.436,-31.429,5.788,RPul1,0
258,RPul-02,Pul,0.989,0.989,0.004,9.537,-41.899,54.743,12.104,-31.392,6.145,RPul2,0
259,RPul-03,Pul,0.990,0.990,0.004,12.931,-41.403,55.108,15.826,-30.915,6.525,RPul3,0
260,RPul-04,Pul,0.992,0.992,0.002,16.436,-40.892,55.443,19.763,-30.358,6.935,RPul4,0
261,RPul-05,Pul,0.991,0.991,0.003,19.769,-40.427,55.647,23.891,-29.832,7.316,RPul5,0
262,RPul-06,Pul,0.980,0.980,0.001,23.189,-40.091,56.214,28.360,-28.992,8.053,RPul6,0
263,RPul-07,Insula,0.990,0.990,0.003,26.561,-39.572,56.261,32.671,-27.585,8.140,RPul7,0


In [40]:
ied_ch_names = [str(ch) for ch in ied["ch_names"]]
channel_renames= [{"name": "RPuI", "new_name": "RPul"}]
    
for channel_name in channel_renames:
    matched = [(idx,ch) for idx,ch in enumerate(ied_ch_names) if ch.startswith(channel_name["name"])]
    for idx,ch in matched:
            new_ch_name = ch.replace(channel_name["name"], channel_name["new_name"])
            ied_ch_names[idx] = new_ch_name


In [41]:
    
wrong_channel_names = []
spike_counts = dict(zip(ied_ch_names, ied["channel_spike_counts"]))
spike_rates = dict(zip(ied_ch_names, ied["channel_spike_rates"]))
df["spike_count"] = df["_normalized"].map(spike_counts)
df["spike_rate"] = df["_normalized"].map(spike_rates)

In [42]:
df[210:230]

,label,atlas_label,GM,WM,CSF,x,y,z,mni_x,mni_y,mni_z,_normalized,is_bad,spike_count,spike_rate
252,RANT-14,Superior Temporal,0.990,0.990,0.004,46.882,-17.208,45.733,53.114,-6.295,-1.322,RANT14,0,126,1.388597
253,RANT-15,Superior Temporal,0.907,0.907,0.001,50.406,-17.637,44.458,57.367,-6.669,-2.752,RANT15,0,325,3.581698
254,RANT-16,Superior Temporal,0.958,0.958,0.025,53.766,-17.687,43.414,62.217,-7.537,-4.536,RANT16,0,77,0.848587
257,RPul-01,Pul,0.985,0.985,0.007,6.062,-42.450,54.509,8.436,-31.429,5.788,RPul1,0,255,2.810255
258,RPul-02,Pul,0.989,0.989,0.004,9.537,-41.899,54.743,12.104,-31.392,6.145,RPul2,0,360,3.967419
259,RPul-03,Pul,0.990,0.990,0.004,12.931,-41.403,55.108,15.826,-30.915,6.525,RPul3,0,385,4.242935
260,RPul-04,Pul,0.992,0.992,0.002,16.436,-40.892,55.443,19.763,-30.358,6.935,RPul4,0,403,4.441306
261,RPul-05,Pul,0.991,0.991,0.003,19.769,-40.427,55.647,23.891,-29.832,7.316,RPul5,0,373,4.110687
262,RPul-06,Pul,0.980,0.980,0.001,23.189,-40.091,56.214,28.360,-28.992,8.053,RPul6,0,270,2.975565
263,RPul-07,Insula,0.990,0.990,0.003,26.561,-39.572,56.261,32.671,-27.585,8.140,RPul7,0,96,1.057979
